# Loan Approval Prediction Application

## Phase 7 – Prediction Interface and Model Integration

**Name:** Abdur Rafay Hassan Baloch  
**Model:** Logistic Regression  
**Application Framework:** Gradio  
**Prediction Output:** Approval Probability and Classification

## 1. Install Application Requirements

Gradio is installed to create an interactive prediction interface in Google Colab.

In [1]:
!pip install gradio -q

## 2. Load the Trained Logistic Regression Model

The previously trained machine learning pipeline is loaded for new loan predictions.

In [3]:
import pandas as pd
import numpy as np
import joblib
import gradio as gr

model = joblib.load("loan_approval_logistic_model.pkl")

feature_columns = [
    "Gender",
    "Married",
    "Dependents",
    "Education",
    "Self_Employed",
    "ApplicantIncome",
    "CoapplicantIncome",
    "LoanAmount",
    "Loan_Amount_Term",
    "Credit_History",
    "Property_Area_Rural",
    "Property_Area_Semiurban",
    "Property_Area_Urban"
]

print("MODEL INTEGRATION DETAILS")
print("-" * 50)
print("Model File: loan_approval_logistic_model.pkl")
print("Model Type: Logistic Regression Pipeline")
print("Model Loading Status: Successfully Loaded")
print("Number of Required Input Features:", len(feature_columns))
print("Prediction Classes: Rejected (0), Approved (1)")
print("Decision Threshold: 50%")

print("\nRequired Features:")
for number, feature in enumerate(feature_columns, start=1):
    print(f"{number}. {feature}")

MODEL INTEGRATION DETAILS
--------------------------------------------------
Model File: loan_approval_logistic_model.pkl
Model Type: Logistic Regression Pipeline
Model Loading Status: Successfully Loaded
Number of Required Input Features: 13
Prediction Classes: Rejected (0), Approved (1)
Decision Threshold: 50%

Required Features:
1. Gender
2. Married
3. Dependents
4. Education
5. Self_Employed
6. ApplicantIncome
7. CoapplicantIncome
8. LoanAmount
9. Loan_Amount_Term
10. Credit_History
11. Property_Area_Rural
12. Property_Area_Semiurban
13. Property_Area_Urban


## 3. Prediction Function

The prediction function converts user input into the same numerical format used during model training.

In [4]:
def predict_loan(
    gender,
    married,
    dependents,
    education,
    self_employed,
    applicant_income,
    coapplicant_income,
    loan_amount,
    loan_term,
    credit_history,
    property_area
):
    # Convert text inputs into trained numerical format
    gender_value = 1 if gender == "Male" else 0
    married_value = 1 if married == "Yes" else 0
    education_value = 1 if education == "Graduate" else 0
    self_employed_value = 1 if self_employed == "Yes" else 0
    credit_history_value = 1 if credit_history == "Good" else 0

    property_rural = 1 if property_area == "Rural" else 0
    property_semiurban = 1 if property_area == "Semiurban" else 0
    property_urban = 1 if property_area == "Urban" else 0

    new_applicant = pd.DataFrame([{
        "Gender": gender_value,
        "Married": married_value,
        "Dependents": int(dependents),
        "Education": education_value,
        "Self_Employed": self_employed_value,
        "ApplicantIncome": float(applicant_income),
        "CoapplicantIncome": float(coapplicant_income),
        "LoanAmount": float(loan_amount),
        "Loan_Amount_Term": float(loan_term),
        "Credit_History": credit_history_value,
        "Property_Area_Rural": property_rural,
        "Property_Area_Semiurban": property_semiurban,
        "Property_Area_Urban": property_urban
    }], columns=feature_columns)

    predicted_class = int(model.predict(new_applicant)[0])
    probabilities = model.predict_proba(new_applicant)[0]

    rejection_probability = probabilities[0] * 100
    approval_probability = probabilities[1] * 100

    if predicted_class == 1:
        prediction = "APPROVED"
        explanation = (
            f"The model predicts that this loan application is likely to be "
            f"approved because the approval probability is "
            f"{approval_probability:.2f}%."
        )
    else:
        prediction = "REJECTED"
        explanation = (
            f"The model predicts that this loan application is likely to be "
            f"rejected because the approval probability is "
            f"{approval_probability:.2f}%."
        )

    return (
        prediction,
        round(approval_probability, 2),
        round(rejection_probability, 2),
        explanation
    )

print("Prediction function created successfully.")
print("User inputs will be converted into 13 numerical model features.")
print("The function will return classification and probability results.")

Prediction function created successfully.
User inputs will be converted into 13 numerical model features.
The function will return classification and probability results.


## 4. Loan Prediction Application UI

The interactive application collects applicant information and displays the loan approval probability and classification.

In [8]:
with gr.Blocks(title="Loan Approval Prediction") as app:

    gr.Markdown(
        """
        # Loan Approval Prediction Application

        Enter the applicant's information below. The trained Logistic
        Regression model will calculate the loan approval probability
        and predict whether the application is likely to be approved
        or rejected.

        **Educational Project by Abdur Rafay Hassan Baloch**
        """
    )

    with gr.Row():
        with gr.Column():
            gender = gr.Radio(
                ["Male", "Female"],
                value="Male",
                label="Gender"
            )

            married = gr.Radio(
                ["Yes", "No"],
                value="Yes",
                label="Married"
            )

            dependents = gr.Dropdown(
                ["0", "1", "2", "3"],
                value="0",
                label="Number of Dependents"
            )

            education = gr.Radio(
                ["Graduate", "Not Graduate"],
                value="Graduate",
                label="Education"
            )

            self_employed = gr.Radio(
                ["Yes", "No"],
                value="No",
                label="Self Employed"
            )

            property_area = gr.Dropdown(
                ["Rural", "Semiurban", "Urban"],
                value="Semiurban",
                label="Property Area"
            )

        with gr.Column():
            applicant_income = gr.Number(
                value=5000,
                minimum=0,
                label="Applicant Income"
            )

            coapplicant_income = gr.Number(
                value=1500,
                minimum=0,
                label="Co-applicant Income"
            )

            loan_amount = gr.Number(
                value=150,
                minimum=1,
                label="Loan Amount"
            )

            loan_term = gr.Dropdown(
                [12, 36, 60, 84, 120, 180, 240, 300, 360, 480],
                value=360,
                label="Loan Amount Term"
            )

            credit_history = gr.Radio(
                ["Good", "Poor"],
                value="Good",
                label="Credit History"
            )

    predict_button = gr.Button(
        "Predict Loan Approval",
        variant="primary"
    )

    gr.Markdown("## Prediction Result")

    with gr.Row():
        prediction_output = gr.Textbox(
            label="Loan Prediction"
        )

        approval_output = gr.Number(
            label="Approval Probability (%)"
        )

        rejection_output = gr.Number(
            label="Rejection Probability (%)"
        )

    explanation_output = gr.Textbox(
        label="Prediction Explanation",
        lines=3
    )

    predict_button.click(
        fn=predict_loan,
        inputs=[
            gender,
            married,
            dependents,
            education,
            self_employed,
            applicant_income,
            coapplicant_income,
            loan_amount,
            loan_term,
            credit_history,
            property_area
        ],
        outputs=[
            prediction_output,
            approval_output,
            rejection_output,
            explanation_output
        ]
    )

    gr.Examples(
        examples=[
            [
                "Male", "Yes", "1", "Graduate", "No",
                6000, 2000, 150, 360, "Good", "Semiurban"
            ],
            [
                "Female", "No", "0", "Not Graduate", "Yes",
                2000, 0, 250, 360, "Poor", "Rural"
            ],
            [
                "Male", "Yes", "2", "Graduate", "No",
                5000, 1500, 180, 360, "Good", "Urban"
            ]
        ],
        inputs=[
            gender,
            married,
            dependents,
            education,
            self_employed,
            applicant_income,
            coapplicant_income,
            loan_amount,
            loan_term,
            credit_history,
            property_area
        ],
        label="Example Applicant Test Cases"
    )

app.launch(share=True, debug=True)

Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().


KeyboardInterrupt: 

In [7]:
test_cases = [
    {
        "Test Case": "Applicant 1",
        "Gender": "Male",
        "Married": "Yes",
        "Dependents": "1",
        "Education": "Graduate",
        "Self Employed": "No",
        "Applicant Income": 6000,
        "Coapplicant Income": 2000,
        "Loan Amount": 150,
        "Loan Term": 360,
        "Credit History": "Good",
        "Property Area": "Semiurban"
    },
    {
        "Test Case": "Applicant 2",
        "Gender": "Female",
        "Married": "No",
        "Dependents": "0",
        "Education": "Not Graduate",
        "Self Employed": "Yes",
        "Applicant Income": 2000,
        "Coapplicant Income": 0,
        "Loan Amount": 250,
        "Loan Term": 360,
        "Credit History": "Poor",
        "Property Area": "Rural"
    },
    {
        "Test Case": "Applicant 3",
        "Gender": "Male",
        "Married": "Yes",
        "Dependents": "2",
        "Education": "Graduate",
        "Self Employed": "No",
        "Applicant Income": 5000,
        "Coapplicant Income": 1500,
        "Loan Amount": 180,
        "Loan Term": 360,
        "Credit History": "Good",
        "Property Area": "Urban"
    }
]

multiple_results = []

for case in test_cases:
    result = predict_loan(
        case["Gender"],
        case["Married"],
        case["Dependents"],
        case["Education"],
        case["Self Employed"],
        case["Applicant Income"],
        case["Coapplicant Income"],
        case["Loan Amount"],
        case["Loan Term"],
        case["Credit History"],
        case["Property Area"]
    )

    multiple_results.append({
        "Test Case": case["Test Case"],
        "Credit History": case["Credit History"],
        "Applicant Income": case["Applicant Income"],
        "Loan Amount": case["Loan Amount"],
        "Prediction": result[0],
        "Approval Probability": f"{result[1]:.2f}%",
        "Rejection Probability": f"{result[2]:.2f}%"
    })

results_table = pd.DataFrame(multiple_results)

print("MULTIPLE APPLICANT TEST RESULTS")
print("-" * 60)

results_table

MULTIPLE APPLICANT TEST RESULTS
------------------------------------------------------------


,Test Case,Credit History,Applicant Income,Loan Amount,Prediction,Approval Probability,Rejection Probability
0,Applicant 1,Good,6000,150,APPROVED,88.41%,11.59%
1,Applicant 2,Poor,2000,250,REJECTED,3.24%,96.76%
2,Applicant 3,Good,5000,180,APPROVED,82.14%,17.86%
